# Phase 2: WARL Swarm Optimization
[<-- Return to Main README](../README.md) | [<-- Previous: Problem Statement](./01_Problem_Statement_and_Methodology.ipynb)

This notebook contains the live generative environment. Because wildfires are chaotic and unpredictable, we do not use static historical CSV files. Instead, the `WildfireSwarmEnv` mathematically generates a new, randomized fire spread pattern every single time it runs. The AI must learn to adapt to live chaos.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Computational Device (Hardware): {device}")

# ==========================================
# 1. LIVE GENERATIVE PHYSICS ENVIRONMENT
# ==========================================
class WildfireSwarmEnv(gym.Env):
    def __init__(self, grid_size=8, num_drones=3, max_payload=20, wind_vector=(1, 0)):
        super(WildfireSwarmEnv, self).__init__()
        self.grid_size = grid_size
        self.num_drones = num_drones
        self.max_payload = max_payload
        self.wind_vector = wind_vector 
        
        self.grid = np.zeros((self.grid_size, self.grid_size), dtype=np.int8)
        self.drone_states = np.zeros((self.num_drones, 3), dtype=np.int32)
        self.action_space = spaces.MultiDiscrete([6] * self.num_drones)
        
        obs_dim = (self.grid_size * self.grid_size) + (self.num_drones * 3)
        self.observation_space = spaces.Box(low=0, high=max(self.grid_size, self.max_payload), shape=(obs_dim,), dtype=np.float32)
        self.max_steps = 75
        self.current_step = 0
        
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        self.grid.fill(0)
        center = self.grid_size // 2
        self.grid[center, center] = 1
        
        for i in range(self.num_drones):
            self.drone_states[i] = [0, 0, self.max_payload]
            
        return self._get_obs(), {}
        
    def _get_obs(self):
        grid_flat = self.grid.flatten().astype(np.float32)
        drones_flat = self.drone_states.flatten().astype(np.float32)
        return np.concatenate((grid_flat, drones_flat))
        
    def step(self, actions):
        self.current_step += 1
        reward = 0.0
        
        for i, action in enumerate(actions):
            y, x, payload = self.drone_states[i]
            
            if action == 0 and y > 0: y -= 1
            elif action == 1 and y < self.grid_size - 1: y += 1
            elif action == 2 and x > 0: x -= 1
            elif action == 3 and x < self.grid_size - 1: x += 1
            elif action == 4:
                if payload > 0 and self.grid[y, x] == 1:
                    self.grid[y, x] = 2 
                    payload -= 1
                    reward += 20.0 # High reward for extinguishing
                else:
                    reward -= 0.1 # Minor penalty for empty drop
            elif action == 5:
                if y == 0 and x == 0:
                    payload = self.max_payload
                    
            self.drone_states[i] = [y, x, payload]
            
        new_grid = self.grid.copy()
        fire_count = 0
        for y in range(self.grid_size):
            for x in range(self.grid_size):
                if self.grid[y, x] == 1:
                    fire_count += 1
                    neighbors = [(y-1, x), (y+1, x), (y, x-1), (y, x+1)]
                    for ny, nx in neighbors:
                        if 0 <= ny < self.grid_size and 0 <= nx < self.grid_size:
                            if self.grid[ny, nx] == 0:
                                spread_prob = 0.02 # Slower base spread
                                if (ny - y) == self.wind_vector[0] and (nx - x) == self.wind_vector[1]:
                                    spread_prob = 0.1 # Slower wind spread
                                if np.random.rand() < spread_prob:
                                    new_grid[ny, nx] = 1
                                    
        self.grid = new_grid
        reward -= (fire_count * 0.1) # Reduced penalty per fire cell
        
        terminated = False
        if fire_count == 0:
            terminated = True
            reward += 200.0 # Massive reward for total containment
        elif self.current_step >= self.max_steps:
            terminated = True
            
        info = {'fire_count': fire_count}
        return self._get_obs(), reward, terminated, False, info

In [ ]:
class SwarmQNetwork(nn.Module):
    def __init__(self, obs_dim, num_drones):
        super(SwarmQNetwork, self).__init__()
        self.num_actions = 6 ** num_drones
        self.fc = nn.Sequential(
            nn.Linear(obs_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, self.num_actions)
        )
    def forward(self, x): return self.fc(x)

class ReplayBuffer:
    def __init__(self, capacity): self.buffer = deque(maxlen=capacity)
    def push(self, state, action_idx, reward, next_state, done): self.buffer.append((state, action_idx, reward, next_state, done))
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        state, action, reward, next_state, done = map(np.stack, zip(*batch))
        return state, action, reward, next_state, done
    def __len__(self): return len(self.buffer)

def index_to_actions(index, num_drones):
    actions = []
    for _ in range(num_drones):
        actions.append(index % 6)
        index //= 6
    return actions

In [ ]:
def train_swarm_dqn(env, episodes=2000, batch_size=128, gamma=0.99, lr=5e-4):
    obs_dim = env.observation_space.shape[0]
    num_drones = env.num_drones
    num_actions = 6 ** num_drones
    
    q_network = SwarmQNetwork(obs_dim, num_drones).to(device)
    target_network = SwarmQNetwork(obs_dim, num_drones).to(device)
    target_network.load_state_dict(q_network.state_dict())
    
    optimizer = optim.Adam(q_network.parameters(), lr=lr)
    buffer = ReplayBuffer(100000)
    
    epsilon = 1.0
    epsilon_decay = 0.995 # Slower decay for more exploration
    epsilon_min = 0.05
    
    rewards_history = []
    fire_counts_history = []
    
    print("Initiating Decentralized WARL Swarm Training (Teaching the AI Drones)...")
    
    for episode in range(episodes):
        state, _ = env.reset()
        total_reward = 0
        done = False
        
        while not done:
            if random.random() < epsilon:
                action_idx = random.randint(0, num_actions - 1)
            else:
                with torch.no_grad():
                    state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
                    q_values = q_network(state_tensor)
                    action_idx = q_values.argmax().item()
                    
            actions = index_to_actions(action_idx, num_drones)
            next_state, reward, done, _, info = env.step(actions)
            
            buffer.push(state, action_idx, reward, next_state, done)
            state = next_state
            total_reward += reward
            
            if len(buffer) > batch_size:
                s, a, r, s_next, d = buffer.sample(batch_size)
                s = torch.FloatTensor(s).to(device)
                a = torch.LongTensor(a).to(device)
                r = torch.FloatTensor(r).to(device)
                s_next = torch.FloatTensor(s_next).to(device)
                d = torch.FloatTensor(d).to(device)
                
                q_values = q_network(s)
                q_value = q_values.gather(1, a.unsqueeze(1)).squeeze(1)
                
                with torch.no_grad():
                    next_q_values = target_network(s_next)
                    max_next_q_values = next_q_values.max(1)[0]
                    target = r + gamma * max_next_q_values * (1 - d)
                    
                loss = nn.MSELoss()(q_value, target)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
        if episode % 10 == 0:
            target_network.load_state_dict(q_network.state_dict())
            
        epsilon = max(epsilon_min, epsilon * epsilon_decay)
        rewards_history.append(total_reward)
        fire_counts_history.append(info['fire_count'])
        
        if (episode + 1) % 100 == 0:
            print(f"Episode {episode+1}/{episodes} | Total Reward (AI Score): {total_reward:.2f} | Remaining Fire: {info['fire_count']} cells | Epsilon (Randomness): {epsilon:.2f}")
            
    return q_network, rewards_history, fire_counts_history

In [ ]:
# Initialize Live Generative Environment: 8x8 Forest Grid, 3 Drones
env = WildfireSwarmEnv(grid_size=8, num_drones=3, max_payload=20, wind_vector=(1, 1))

# Train Model on Live Data (Increased to 2000 episodes for convergence)
trained_model, rewards, fires = train_swarm_dqn(env, episodes=2000)

# Plot Convergence Metrics (AI Learning Results)
fig, axs = plt.subplots(2, 1, figsize=(10, 8))

axs[0].plot(rewards, color='purple', alpha=0.7)
axs[0].set_title('WARL Swarm Convergence: Total Reward per Episode (AI Learning Score)')
axs[0].set_ylabel('Reward (Score)')
axs[0].grid(True, alpha=0.3)

axs[1].plot(fires, color='red', alpha=0.7)
axs[1].set_title('Suppression Efficacy: Remaining Fire Cells per Episode (Unextinguished Fire)')
axs[1].set_xlabel('Episode (Simulation Run)')
axs[1].set_ylabel('Active Fire Cells (Danger Level)')
axs[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()